# DepthWizard — Phase-1 Feasibility (Colab GPU runner)

**Goal:** test ONE hypothesis before building anything — *can a small learned
RGB+relative-depth fusion head predict object height (nDSM) better than trivial
affine calibration, especially on a **fully held-out city**?*

Depth Anything V2 is used **only as a frozen relative-depth prior**, never as a
metric height sensor. This notebook runs Baselines **A** (raw depth), **B**
(global affine), **C** (small learned fusion head) and an optional **D**
(RDAH-Net reference), reports MAE/RMSE/Pearson (all / building / non-building,
per-scene, in-domain vs cross-city), and prints a **GO / MODIFY / ABANDON**
recommendation computed from the measured numbers.

> ⛔ **This is a checkpoint, not a green light.** The notebook stops after the
> decision. A human reviews the numbers **and the error maps** before any full
> pipeline / 3D flythrough is built.


## 0 · Hardware check


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'no GPU (CPU works but is slow)')
try:
    import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
except Exception as e:
    print('torch not importable yet:', e)


## 1 · Get the code

Either clone the repo, or upload the `DepthWizard/` folder to the Colab file
browser. Edit the path below to match.  (If you `git init`+push this project to
your own GitHub, replace the URL.)


In [ ]:
import os
REPO_DIR = 'DepthWizard'
# Option A: clone your fork (uncomment + set URL)
# !git clone https://github.com/<you>/DepthWizard.git
# Option B: you uploaded the folder already — just point at it.
assert os.path.isdir(REPO_DIR), f'Put the project at ./{REPO_DIR} (clone or upload) and re-run.'
os.chdir(REPO_DIR); print('cwd =', os.getcwd())


## 2 · Install dependencies
On Colab, torch is preinstalled; the rest are quick.


In [ ]:
!pip install -q -r requirements.txt


## 3 · Self-check (fast, ~seconds) — NOT evidence

Runs the **entire A/B/C pipeline on tiny synthetic data with a fake depth stub**,
including training the learned head, so any code/runtime bug surfaces here in
seconds — *before* you spend GPU time on the real dataset. The report it writes
is clearly banner-marked **NOT VALID EVIDENCE**.


In [ ]:
!python scripts/run_phase1.py --config configs/smoke.yaml --allow-fake-depth --out runs/smoke


## 4 · Acquire DFC2019 (real evidence)

DFC2019 Track-1 (US3D) tiles must sit under `data/dfc2019/` in the canonical
layout `*_RGB.tif` / `*_AGL.tif` / `*_CLS.tif`, named by city (`JAX_*`, `OMA_*`).
Pick ONE:

- **Kaggle** (easiest): add a DFC2019 / US3D dataset to the notebook, then symlink
  or copy the tiles into `data/dfc2019/`.
- **IEEE DataPort**: https://ieee-dataport.org/open-access/data-fusion-contest-2019-dfc2019 (login + EULA).
- **HF mirror**: set `data.source: hf_mirror` in `configs/phase1.yaml` (schema is
  inspected at runtime; verify it produced real triplets).

Then confirm the tiles are discoverable and the configured cities are present:


In [ ]:
# e.g. Kaggle:  !kaggle datasets download -d <owner>/<dfc2019-slug> -p data/ --unzip
# then make sure files land at data/dfc2019/<CITY>_*_RGB.tif etc.
!python scripts/00_fetch_dataset.py configs/phase1.yaml


## 5 · Run the REAL Phase-1 experiment

City-held-out: train on `JAX`, validate in-domain on held-out `JAX` tiles, and
report the headline number on the **never-seen `OMA`** city. Adjust cities /
caps / epochs in `configs/phase1.yaml`. Depth-prior outputs are cached to disk,
so re-runs are fast.


In [ ]:
!python scripts/run_phase1.py --config configs/phase1.yaml


## 6 · Read the results, decision, and error maps


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('EXPERIMENT_RESULTS.md', encoding='utf-8').read()))


In [ ]:
import glob
from IPython.display import Image, display
figs = sorted(glob.glob('runs/phase1/figures/*.png'))
print(f'{len(figs)} figures'); [display(Image(f)) for f in figs[:8]]


## 7 · ⛔ CHECKPOINT — stop here

Read the **verdict** and, just as important, the **error maps** above.

- **GO** — learned fusion beat affine on the held-out city and is within/near the
  reference band. Proceed to build the full pipeline around this (swappable) head.
- **MODIFY** — the prior carries signal but the head doesn't beat trivial
  calibration cross-city (or fails a guardrail). Iterate before scaling.
- **ABANDON** — the frozen depth prior carries ~no monotone height signal; the
  core premise fails on this data.

Remember the two things this experiment **cannot** tell you: performance on
**Indian ISRO imagery** (domain shift — the top risk), and full-**DSM** accuracy
(this measures nDSM only; adding a DEM/DTM compounds its own error). A human
makes the final call.
